In [0]:
dbutils.widgets.text("catalog", "fmcg")
catalog = dbutils.widgets.get("catalog")

spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.{schema}.vw_fact_orders_enriched AS (
        SELECT 
            fo.date,
            fo.product_code,
            fo.customer_code,

            -- Date attributes
            dd.date_key,
            dd.year,
            dd.month_name,
            dd.month_short_name,
            dd.quarter,
            dd.year_quarter,

            -- Customer attributes
            dc.customer,
            dc.market,
            dc.platform,
            dc.channel,

            -- Product attributes
            dp.division,
            dp.category,
            dp.product,
            dp.variant,

            -- Metrics
            fo.sold_quantity,
            gp.price_inr,

            -- Derived Metric: Amount
            (fo.sold_quantity * gp.price_inr) AS total_amount_inr
        
        FROM {catalog}.{schema}.fact_orders fo

        -- Join with Date Dimension
        LEFT JOIN {catalog}.{schema}.dim_date dd
               ON fo.date = dd.month_start_date

        -- Join with Customers
        LEFT JOIN {catalog}.{schema}.dim_customers dc 
               ON fo.customer_code = dc.customer_code

        -- Join with Products
        LEFT JOIN {catalog}.{schema}.dim_products dp 
               ON fo.product_code = dp.product_code

        -- Join with Price (year-based)
        LEFT JOIN {catalog}.{schema}.dim_gross_price gp 
               ON fo.product_code = gp.product_code
              AND YEAR(fo.date) = gp.year
    )
""")

# Preview the view
display(spark.sql(f"SELECT * FROM {catalog}.{schema}.vw_fact_orders_enriched"))